In [ ]:
pip install


In [ ]:
from kubernetes import client as k8s_client
from kubeflow.training import V1ReplicaSpec, V1RunPolicy
from kubeflow.training import client as training_client
from kubernetes.client.models import V1ObjectMeta, V1PodSpec, V1PodTemplateSpec
from kubeflow.training import V1TFJob, V1TFJobSpec, V1PyTorchJob, V1PyTorchJobSpec, V1MPIJob, V1MPIJobSpec

class TrainingJobSDK:
    def __init__(self, name, namespace, container_name, training_image, model_dir, data_dir, local_output_dir, script_dir, resources=None):
        self.name = name
        self.namespace = namespace
        self.container_name = container_name
        self.training_image = training_image
        self.model_dir = model_dir
        self.data_dir = data_dir
        self.local_output_dir = local_output_dir
        self.script_dir = script_dir
        self.client = training_client.TrainingClient(namespace=self.namespace)

        # 默认资源配置：允许用户覆盖
        self.resources = resources or {
            "limits": {"cpu": "2", "memory": "4Gi"},
            "requests": {"cpu": "1", "memory": "2Gi"}
        }

    def create_volume(self, name, host_path):
        """创建 Kubernetes 的 Volume"""
        return k8s_client.V1Volume(
            name=name,
            host_path=k8s_client.V1HostPathVolumeSource(path=host_path)
        )

    def create_volume_mount(self, name, mount_path):
        """创建 VolumeMount 以挂载到容器中"""
        return k8s_client.V1VolumeMount(name=name, mount_path=mount_path)

    def create_container(self, script_name, args):
        """创建容器配置，包含资源定义"""
        return k8s_client.V1Container(
            name=self.container_name,
            image=self.training_image,
            command=["python", f"{self.script_dir}/{script_name}"] + args,
            volume_mounts=self.create_volume_mounts(),
            resources=k8s_client.V1ResourceRequirements(
                limits=self.resources.get("limits"),
                requests=self.resources.get("requests")
            )
        )

    def create_volume_mounts(self):
        """生成所有需要挂载的 VolumeMount 列表"""
        return [
            self.create_volume_mount("model-volume", self.model_dir),
            self.create_volume_mount("data-volume", self.data_dir),
            self.create_volume_mount("local-output-volume", self.local_output_dir),
            self.create_volume_mount("script-volume", self.script_dir)
        ]

    def create_volumes(self):
        """生成所有需要的 Volume 列表"""
        return [
            self.create_volume("model-volume", self.model_dir),
            self.create_volume("data-volume", self.data_dir),
            self.create_volume("local-output-volume", self.local_output_dir),
            self.create_volume("script-volume", self.script_dir)
        ]

    def create_pod_spec(self, script_name, args):
        """创建 PodSpec 配置"""
        container = self.create_container(script_name, args)
        return V1PodSpec(containers=[container], volumes=self.create_volumes())

    def create_replica_spec(self, replicas, pod_spec):
        """创建 ReplicaSpec 配置"""
        return V1ReplicaSpec(
            replicas=replicas,
            restart_policy="Never",
            template=V1PodTemplateSpec(spec=pod_spec)
        )

    def create_training_job(self, job_type, replicas, script_name, args):
        """创建不同类型的训练作业"""
        pod_spec = self.create_pod_spec(script_name, args)

        if job_type == "TFJob":
            return self._create_tf_job(replicas, pod_spec)
        elif job_type == "PyTorchJob":
            return self._create_pytorch_job(replicas, pod_spec)
        elif job_type == "MPIJob":
            return self._create_mpi_job(replicas, pod_spec)
        else:
            raise ValueError(f"Unsupported job type: {job_type}")

    def _create_tf_job(self, replicas, pod_spec):
        """私有方法：创建 TFJob"""
        return V1TFJob(
            api_version="kubeflow.org/v1",
            kind="TFJob",
            metadata=V1ObjectMeta(name=self.name, namespace=self.namespace),
            spec=V1TFJobSpec(
                run_policy=V1RunPolicy(clean_pod_policy="None"),
                tf_replica_specs={
                    "Chief": self.create_replica_spec(replicas.get("Chief", 1), pod_spec),
                    "Worker": self.create_replica_spec(replicas.get("Worker", 2), pod_spec),
                    "PS": self.create_replica_spec(replicas.get("PS", 1), pod_spec)
                }
            )
        )

    def _create_pytorch_job(self, replicas, pod_spec):
        """私有方法：创建 PyTorchJob"""
        return V1PyTorchJob(
            api_version="kubeflow.org/v1",
            kind="PyTorchJob",
            metadata=V1ObjectMeta(name=self.name, namespace=self.namespace),
            spec=V1PyTorchJobSpec(
                run_policy=V1RunPolicy(clean_pod_policy="None"),
                pytorch_replica_specs={
                    "Master": self.create_replica_spec(replicas.get("Master", 1), pod_spec),
                    "Worker": self.create_replica_spec(replicas.get("Worker", 2), pod_spec)
                }
            )
        )

    def _create_mpi_job(self, replicas, pod_spec):
        """私有方法：创建 MPIJob"""
        return V1MPIJob(
            api_version="kubeflow.org/v1",
            kind="MPIJob",
            metadata=V1ObjectMeta(name=self.name, namespace=self.namespace),
            spec=V1MPIJobSpec(
                run_policy=V1RunPolicy(clean_pod_policy="None"),
                mpi_replica_specs={
                    "Launcher": self.create_replica_spec(replicas.get("Launcher", 1), pod_spec),
                    "Worker": self.create_replica_spec(replicas.get("Worker", 2), pod_spec)
                }
            )
        )

    def submit_job(self, job):
        """提交训练作业"""
        if isinstance(job, (V1TFJob, V1PyTorchJob, V1MPIJob)):
            self.client.create_job(job, job.metadata.name)
        else:
            raise ValueError("Unsupported job type for submission")

    def get_job_status(self):
        """获取训练作业的状态"""
        try:
            status = self.client.get_job(self.name)
            print(f"Job status: {status.status.conditions}")
        except Exception as e:
            print(f"Failed to get job status: {e}")

    def get_job_logs(self, follow=False, verbose=False):
        """获取训练作业的日志"""
        try:
            logs, events = self.client.get_job_logs(
                name=self.name,
                follow=follow,
                verbose=verbose
            )
            for pod_name, log in logs.items():
                print(f"Logs for {pod_name}: {log}")
            if verbose:
                for event_name, event_data in events.items():
                    print(f"Events for {event_name}: {event_data}")
        except Exception as e:
            print(f"Failed to get job logs: {e}")

    def update_job(self, updated_job):
        """更新训练作业"""
        try:
            self.client.update_job(updated_job, name=self.name)
            print(f"Job '{self.name}' updated successfully.")
        except Exception as e:
            print(f"Failed to update job: {e}")

    def delete_job(self):
        """删除训练作业"""
        try:
            self.client.delete_job(name=self.name)
            print(f"Job '{self.name}' deleted successfully.")
        except Exception as e:
            print(f"Failed to delete job: {e}")

    def get_job_pods(self):
        """获取训练作业相关的 Pod 列表"""
        try:
            pods = self.client.get_job_pods(name=self.name)
            for pod in pods:
                print(f"Pod name: {pod.metadata.name}, Status: {pod.status.phase}")
        except Exception as e:
            print(f"Failed to get job pods: {e}")



In [ ]:
# 使用示例
sdk = TrainingJobSDK(
    name="example-tfjob",
    namespace="kubeflow-user-example-com",
    container_name="training-container",
    training_image="tensorflow/tensorflow:2.4.1",
    model_dir="/mnt/model",
    data_dir="/mnt/data",
    local_output_dir="/mnt/local_output",
    script_dir="/mnt/scripts"
)

# 定义 TFJob
job = sdk.create_training_job(
    job_type="TFJob",
    replicas={"Chief": 1, "Worker": 2, "PS": 1},
    script_name="mnist_with_summaries.py",
    args=[
        "--log_dir=/mnt/model/logs",
        "--data_dir=/mnt/data",
        "--model_dir=/mnt/model",
        "--learning_rate=0.01",
        "--batch_size=150"
    ]
)


In [ ]:
# 使用示例
sdk = TrainingJobSDK(
    name="example-pytorchjob",
    namespace="kubeflow-user-example-com",
    container_name="training-container",
    training_image="pytorch/pytorch:1.7.1",
    model_dir="/mnt/model",
    data_dir="/mnt/data",
    local_output_dir="/mnt/local_output",
    script_dir="/mnt/scripts"
)

# 定义 PyTorchJob
job = sdk.create_training_job(
    job_type="PyTorchJob",
    replicas={"Master": 1, "Worker": 2},
    script_name="mnist_pytorch.py",
    args=[
        "--epochs=10",
        "--batch_size=64",
        "--learning_rate=0.01",
        "--model_dir=/mnt/model"
    ]
)


In [ ]:
'''
Author: ChZheng
Date: 2024-08-23 17:06:15
LastEditTime: 2024-09-02 19:25:43
LastEditors: ChZheng
Description:
FilePath: /笔记/Users/apple/go/src/github.com/training-operator/examples/sdk/train.ipynb
'''
# 使用示例
sdk = TrainingJobSDK(
    name="example-mpijob",
    namespace="kubeflow-user-example-com",
    container_name="training-container",
    training_image="horovod/horovod:0.21.0-tf2.4.1-torch1.7.1",
    model_dir="/mnt/model",
    data_dir="/mnt/data",
    local_output_dir="/mnt/local_output",
    script_dir="/mnt/scripts"
)

# 定义 MPIJob
job = sdk.create_training_job(
    job_type="MPIJob",
    replicas={"Launcher": 1, "Worker": 4},
    script_name="train_horovod.py",
    args=[
        "--epochs=5",
        "--batch_size=128",
        "--learning_rate=0.001",
        "--data_dir=/mnt/data",
        "--model_dir=/mnt/model"
    ]
)




In [ ]:
# 提交 MPIJob
sdk.submit_job(job)


In [ ]:
# 获取 MPIJob 日志
sdk.get_job_logs(follow=True, verbose=True)


In [ ]:
# 删除 MPIJob
sdk.delete_job()


In [ ]:
# 获取 MPIJob 状态
sdk.get_job_status()


In [ ]:
# 获取 MPIJob 的 Pod 列表
sdk.get_job_pods()
